# 49. Protein Docking Automation

## Purpose
Automate the repetitive parts of the docking pipeline (PDB download, receptor prep,
ligand prep, Vina execution, caching) into `src/tools/docking.py`, while keeping
cofactor/HETATM decisions as an explicit human-confirmation step (never auto-decided).
Eventually feed docking score deltas into the debate-based verifier (notebook 48) as
quantitative evidence.

## Background
- Preliminary proposal submitted. Now in the finals prep period.
- Notebooks 45-47: `Aliphatic_long_chain` rule fixed, `PRECEDENT_LIBRARY` expanded 11→23
  entries (all 10 priority rules covered).
- Notebook 48: LLM call parallelization (`batch_iterative_fix_loop`) + debate-based
  verifier orchestration (`ask_llm_debate_fix`, proposer-critic multi-round debate,
  triggered only on candidates with a "[참고]" caution note, budget-capped, mock-tested).
  Hooked into `iterative_fix_loop` via `use_debate` flag, no regressions.
- Qwen (`qwen3.8-max`, DashScope token-plan) weekly quota exhausted, resets 8/15 15:37 UTC —
  debate logic verified with a mock client only so far, needs real-API verification once
  quota resets.
- Existing docking precedents (COMT/EGFR/NQO1) were done manually per-target in earlier
  sessions; this notebook turns that into a reusable function (`auto_dock_precedent`) with
  a `DOCKING_TARGETS` registry, explicit `keep_hetatm_codes` requirement, and a JSON cache.

## This session's goals
1. Install AutoDock Vina + Open Babel, verify `docking.py` runs end-to-end.
2. Reproduce the earlier manual COMT result (dopamine vs methoxydopamine) to confirm the
   automation matches known-good numbers.
3. Once validated, plan connecting docking deltas into the debate verifier's critic prompt.

## Working style preferences
- For changes under 20 lines that aren't new files/entries: describe the insertion point
  instead of a full code block.
- Always provide reload + verification code, combined into a single cell.
- After any file edit: syntax-check → reload → verify → commit immediately.

In [1]:
# 셀1 - install
!pip install rdkit -q
!pip install chembl_webresource_client -q
!pip install fuzzywuzzy python-Levenshtein -q
!pip install PyTDC --no-deps -q
!pip install PyYAML tqdm requests -q
!pip install openai -q
!apt-get install -y openbabel -qq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.4/37.4 MB 38.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.2/55.2 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.8/70.8 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 157.6/157.6 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 62.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.2/154.2 kB 8.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
Selecting previously unselected package libboost-iostreams1.74.0:amd64.
(Reading database ... 118332 files and directories currently installed.)
Preparing to unpack .../libboost-iostreams1.74.0_1.74.0-14ubuntu3_amd64.deb ...
Unpacking libboost-iostreams1.74.0:amd64 (1.74.0-14ubuntu3) ...
Selecting previously unselected package libinchi1.
Preparing to unpack .../libinchi1_1.03+dfsg-4_amd64.deb ...
Unpacki

In [2]:
# 셀2 - github token + clone + cd + pwd
from google.colab import userdata

token = userdata.get('GITHUB_TOKEN')
!git clone https://{token}@github.com/Dec32th/laidd-2026.git
%cd /content/laidd-2026
!pwd

Cloning into 'laidd-2026'...
remote: Enumerating objects: 698, done.
remote: Counting objects: 100% (158/158), done.
remote: Compressing objects: 100% (111/111), done.
remote: Total 698 (delta 82), reused 115 (delta 47), pack-reused 540 (from 1)
Receiving objects: 100% (698/698), 7.16 MiB | 15.44 MiB/s, done.
Resolving deltas: 100% (399/399), done.
/content/laidd-2026
/content/laidd-2026


In [3]:
# 셀3 - git config
!git config --global user.email "hyekyeong.w@gmail.com"
!git config --global user.name "Dec32th"

In [4]:
# 셀4 - import + 데이터 로드 + vina 설치
import importlib, json, ast, time, os
from collections import Counter
from rdkit import Chem
from chembl_webresource_client.new_client import new_client
import requests

import src.tools.replacement_library
import src.tools.molecule_editor
import src.tools.atom_editor
import src.tools.toxicophore_detector
import src.tools.precedent_library
import src.tools.agent

from src.tools.data_prep import load_tox21_clean
from src.tools.toxicophore_detector import detect_toxicophores
from src.tools.replacement_library import get_replacement_candidates
from src.tools.molecule_editor import propose_fix, iterative_fix_loop, clear_failure_memory
from src.tools.precedent_library import PRECEDENT_LIBRARY, get_precedents

data = load_tox21_clean(random_state=7)
molecule = new_client.molecule

base_url = "https://www.guidetopharmacology.org/services"
def search_ligand(name):
    resp = requests.get(f"{base_url}/ligands", params={"name": name})
    return resp.json()
def get_ligand_interactions(ligand_id):
    resp = requests.get(f"{base_url}/ligands/{ligand_id}/interactions")
    return resp.json()

!wget -q https://github.com/ccsb-scripps/AutoDock-Vina/releases/download/v1.2.5/vina_1.2.5_linux_x86_64 -O vina_bin
!chmod +x vina_bin
print("vina_bin 존재:", os.path.exists('vina_bin'))

print(f"선례 수: {len(PRECEDENT_LIBRARY)} (23이어야 정상)")
print("agent.py 토의 로직 반영 여부:", 'ask_llm_debate_fix' in open('src/tools/agent.py').read())
print("molecule_editor.py use_debate 반영 여부:", 'use_debate' in open('src/tools/molecule_editor.py').read())

[05:49:22] WARNING: not removing hydrogen atom without neighbors
[05:49:22] Explicit valence for atom # 8 Al, 6, is greater than permitted
[05:49:23] Explicit valence for atom # 3 Al, 6, is greater than permitted
[05:49:23] Explicit valence for atom # 4 Al, 6, is greater than permitted
[05:49:23] Explicit valence for atom # 4 Al, 6, is greater than permitted
[05:49:23] Explicit valence for atom # 9 Al, 6, is greater than permitted
[05:49:23] Explicit valence for atom # 5 Al, 6, is greater than permitted
[05:49:23] Explicit valence for atom # 16 Al, 6, is greater than permitted
[05:49:24] Explicit valence for atom # 20 Al, 6, is greater than permitted
[05:49:24] WARNING: not removing hydrogen atom without neighbors


전체: 7831개, 파싱 성공: 7823개, 파싱 실패(제외): 8개
vina_bin 존재: True
선례 수: 23 (23이어야 정상)
agent.py 토의 로직 반영 여부: True
molecule_editor.py use_debate 반영 여부: True


In [18]:
%%writefile src/tools/docking.py
"""도킹 자동화 파이프라인.
표적별 HETATM 처리(보조인자 유지 여부)는 반드시 사람이 먼저 확인해야
하므로, 새 표적을 추가할 때는 DOCKING_TARGETS에 keep_hetatm_codes를
명시적으로 등록하는 과정을 거친다(완전 자동화 금지 지점)."""

import os
import json
import re
import subprocess
from rdkit import Chem
from rdkit.Chem import AllChem

VINA_BIN = os.path.join(os.getcwd(), "vina_bin")

DOCKING_TARGETS = {
    "catechol": {
        "target_name": "COMT", "pdb_id": "1VID",
        "ligand_code": "DNC", "keep_hetatm_codes": ["MG", "SAM"],
        "box_size": [20, 20, 20],
    },
    "Michael_acceptor_1": {
        "target_name": "EGFR", "pdb_id": "6JX4",
        "ligand_code": "YY3", "keep_hetatm_codes": [],
        "box_size": [20, 20, 20],
    },
    "hydroquinone": {
        "target_name": "NQO1", "pdb_id": "1DXO",
        "ligand_code": "DQN", "keep_hetatm_codes": ["FAD"],
        "box_size": [20, 20, 20],
    },
    "quinone_A(370)": {
        "target_name": "NQO1", "pdb_id": "1DXO",
        "ligand_code": "DQN", "keep_hetatm_codes": ["FAD"],
        "box_size": [20, 20, 20],
    },
}

_CACHE_PATH = "outputs/docking_cache.json"


def _load_cache():
    if os.path.exists(_CACHE_PATH):
        with open(_CACHE_PATH) as f:
            return json.load(f)
    return {}


def _save_cache(cache):
    os.makedirs("outputs", exist_ok=True)
    with open(_CACHE_PATH, "w") as f:
        json.dump(cache, f, ensure_ascii=False, indent=2)


def prepare_ligand_pdbqt(smiles, filename, seed=42):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    mol = Chem.AddHs(mol)
    if AllChem.EmbedMolecule(mol, randomSeed=seed) != 0:
        return None
    AllChem.MMFFOptimizeMolecule(mol)
    Chem.MolToPDBFile(mol, f"{filename}.pdb")
    subprocess.run(["obabel", f"{filename}.pdb", "-O", f"{filename}.pdbqt"], capture_output=True)
    out_path = f"{filename}.pdbqt"
    return out_path if os.path.exists(out_path) else None


def run_docking_cli(ligand_pdbqt, receptor_pdbqt, out_prefix, box_center, box_size, exhaustiveness=4):
    log_path = f"{out_prefix}_log.txt"
    cmd = [
        VINA_BIN, "--receptor", receptor_pdbqt, "--ligand", ligand_pdbqt,
        "--center_x", str(box_center[0]), "--center_y", str(box_center[1]), "--center_z", str(box_center[2]),
        "--size_x", str(box_size[0]), "--size_y", str(box_size[1]), "--size_z", str(box_size[2]),
        "--exhaustiveness", str(exhaustiveness), "--out", f"{out_prefix}_out.pdbqt",
    ]
    try:
        with open(log_path, "w") as log_f:
            subprocess.run(cmd, stdout=log_f, stderr=subprocess.STDOUT, timeout=180)
    except subprocess.TimeoutExpired:
        return None

    if not os.path.exists(log_path):
        return None
    with open(log_path) as f:
        log = f.read()
    match = re.search(r"^\s*1\s+(-?\d+\.\d+)", log, re.MULTILINE)
    return float(match.group(1)) if match else None


def prepare_receptor(pdb_id, keep_hetatm_codes):
    """수용체 준비. keep_hetatm_codes는 반드시 사람이 그 표적의 HETATM
    목록(prepare_receptor 호출 전 raw PDB를 먼저 열어 확인)을 보고
    직접 지정해야 하며, 빈 리스트라도 명시적으로 넘겨야 한다."""
    os.makedirs("targets", exist_ok=True)
    raw_path = f"targets/{pdb_id}.pdb"
    if not os.path.exists(raw_path):
        subprocess.run(["wget", "-q", f"https://files.rcsb.org/download/{pdb_id}.pdb", "-O", raw_path])

    with open(raw_path) as f:
        lines = f.readlines()

    keep_set = set(keep_hetatm_codes)
    clean_lines = [l for l in lines if l.startswith(("ATOM", "TER", "END"))
                   or (l.startswith("HETATM") and l[17:20].strip() in keep_set)]
    clean_path = f"targets/{pdb_id}_clean.pdb"
    with open(clean_path, "w") as f:
        f.writelines(clean_lines)

    receptor_pdbqt = f"targets/{pdb_id}_receptor.pdbqt"
    if not os.path.exists(receptor_pdbqt):
        subprocess.run(["obabel", clean_path, "-O", receptor_pdbqt, "-xr"], capture_output=True)
    return receptor_pdbqt, lines


def inspect_hetatm(pdb_id):
    """새 표적을 DOCKING_TARGETS에 등록하기 전, HETATM 목록을 먼저
    확인하는 용도. 자동 결정 없이 사람이 보고 keep_hetatm_codes를
    정하도록 정보만 제공한다."""
    os.makedirs("targets", exist_ok=True)
    raw_path = f"targets/{pdb_id}.pdb"
    if not os.path.exists(raw_path):
        subprocess.run(["wget", "-q", f"https://files.rcsb.org/download/{pdb_id}.pdb", "-O", raw_path])
    with open(raw_path) as f:
        lines = f.readlines()
    hetero = set(l[17:20].strip() for l in lines if l.startswith("HETATM"))
    return hetero


def auto_dock_precedent(rule_name, original_smiles, fixed_smiles, use_cache=True):
    """rule_name으로 DOCKING_TARGETS에서 표적 정보를 찾아 도킹 실행.
    pdb_id/ligand_code가 None이면 아직 사람 확인이 안 된 표적이므로
    명확히 에러를 반환한다(자동으로 대충 진행하지 않음)."""
    if rule_name not in DOCKING_TARGETS:
        return {"error": f"'{rule_name}'은 DOCKING_TARGETS에 등록되지 않음"}

    target_info = DOCKING_TARGETS[rule_name]
    if target_info["pdb_id"] is None or target_info["ligand_code"] is None:
        return {"error": f"'{rule_name}' 표적의 pdb_id/ligand_code가 아직 "
                          f"확인 안 됨. inspect_hetatm()으로 먼저 확인 후 "
                          f"DOCKING_TARGETS를 채워주세요."}

    cache = _load_cache() if use_cache else {}
    cache_key = f"{rule_name}|{original_smiles}|{fixed_smiles}"
    if use_cache and cache_key in cache:
        return cache[cache_key]

    if not os.path.exists(VINA_BIN):
        return {"error": f"vina_bin이 {VINA_BIN}에 없음. 다운로드 먼저 진행하세요."}

    receptor_pdbqt, lines = prepare_receptor(target_info["pdb_id"], target_info["keep_hetatm_codes"])

    ligand_lines = [l for l in lines if l.startswith("HETATM") and l[17:20].strip() == target_info["ligand_code"]]
    if not ligand_lines:
        return {"error": f"리간드 코드 {target_info['ligand_code']}를 PDB에서 찾을 수 없음"}
    coords = [(float(l[30:38]), float(l[38:46]), float(l[46:54])) for l in ligand_lines]
    box_center = [sum(c[i] for c in coords) / len(coords) for i in range(3)]

    scores = {}
    for label, smi in [("original", original_smiles), ("fixed", fixed_smiles)]:
        lig_pdbqt = prepare_ligand_pdbqt(smi, f"targets/{rule_name}_{label}")
        if lig_pdbqt is None:
            return {"error": f"{label} 리간드 준비 실패"}
        scores[label] = run_docking_cli(lig_pdbqt, receptor_pdbqt, f"targets/{rule_name}_{label}",
                                          box_center, target_info["box_size"])

    result = {
        "target": target_info["target_name"], "pdb_id": target_info["pdb_id"], "rule": rule_name,
        "score_original": scores["original"], "score_fixed": scores["fixed"],
        "delta": (scores["fixed"] - scores["original"])
                 if scores["original"] is not None and scores["fixed"] is not None else None
    }

    if use_cache:
        cache[cache_key] = result
        _save_cache(cache)

    return result

Overwriting src/tools/docking.py


In [6]:
import ast
with open('src/tools/docking.py') as f:
    ast.parse(f.read())
print("✅ docking.py 문법 정상")

import src.tools.docking
importlib.reload(src.tools.docking)
from src.tools.docking import auto_dock_precedent, inspect_hetatm, DOCKING_TARGETS
print("✅ import 성공, 등록된 표적:", list(DOCKING_TARGETS.keys()))

✅ docking.py 문법 정상
✅ import 성공, 등록된 표적: ['catechol', 'Michael_acceptor_1', 'hydroquinone', 'quinone_A(370)']


In [7]:
dopamine_smiles = "NCCc1ccc(O)c(O)c1"
fixed_dopamine = propose_fix(dopamine_smiles, "catechol", candidate_idx=0)
print(fixed_dopamine['new_smiles'])

COc1ccc(CCN)cc1O


In [8]:
result = auto_dock_precedent(
    "catechol",
    original_smiles="NCCc1ccc(O)c(O)c1",
    fixed_smiles="COc1ccc(CCN)cc1O",
)
print(result)

{'target': 'COMT', 'pdb_id': '1VID', 'rule': 'catechol', 'score_original': -5.592, 'score_fixed': -5.844, 'delta': -0.25200000000000067}


In [9]:
importlib.reload(src.tools.docking)
from src.tools.docking import prepare_receptor, prepare_ligand_pdbqt, run_docking_cli, DOCKING_TARGETS

# 실제로 seed 파라미터가 반영됐는지 먼저 확인
import inspect
print(inspect.signature(prepare_ligand_pdbqt))

(smiles, filename, seed=42)


In [10]:
target_info = DOCKING_TARGETS["catechol"]
receptor_pdbqt, lines = prepare_receptor(target_info["pdb_id"], target_info["keep_hetatm_codes"])
ligand_lines = [l for l in lines if l.startswith("HETATM") and l[17:20].strip() == target_info["ligand_code"]]
coords = [(float(l[30:38]), float(l[38:46]), float(l[46:54])) for l in ligand_lines]
box_center = [sum(c[i] for c in coords) / len(coords) for i in range(3)]
print("box_center:", box_center)

for label, smi in [("dopamine", "NCCc1ccc(O)c(O)c1"), ("methoxydopamine", "COc1ccc(CCN)cc1O")]:
    scores = []
    for seed in [1, 2, 3, 42, 99]:
        lig_pdbqt = prepare_ligand_pdbqt(smi, f"targets/test_{label}_{seed}", seed=seed)
        score = run_docking_cli(lig_pdbqt, receptor_pdbqt, f"targets/test_{label}_{seed}",
                                  box_center, target_info["box_size"], exhaustiveness=16)
        scores.append(score)
    valid = [s for s in scores if s is not None]
    avg = sum(valid) / len(valid) if valid else None
    print(f"{label}: {scores}, 평균={avg}")

box_center: [-24.806714285714285, 59.999857142857145, 50.56657142857143]
dopamine: [-5.622, -5.616, -5.599, -5.728, -5.733], 평균=-5.659599999999999
methoxydopamine: [-5.882, -5.862, -5.853, -5.765, -5.832], 평균=-5.8388


In [11]:
from rdkit import Chem

# 중성 아민 vs 양성자화된 아민(NH3+) 비교
tests = {
    "dopamine_neutral": "NCCc1ccc(O)c(O)c1",
    "dopamine_protonated": "[NH3+]CCc1ccc(O)c(O)c1",
    "methoxydopamine_neutral": "COc1ccc(CCN)cc1O",
    "methoxydopamine_protonated": "COc1ccc(CC[NH3+])cc1O",
}

for label, smi in tests.items():
    mol = Chem.MolFromSmiles(smi)
    print(label, "-> valid:", mol is not None, "| formal charge:", Chem.GetFormalCharge(mol) if mol else None)

dopamine_neutral -> valid: True | formal charge: 0
dopamine_protonated -> valid: True | formal charge: 1
methoxydopamine_neutral -> valid: True | formal charge: 0
methoxydopamine_protonated -> valid: True | formal charge: 1


In [12]:
target_info = DOCKING_TARGETS["catechol"]
receptor_pdbqt, lines = prepare_receptor(target_info["pdb_id"], target_info["keep_hetatm_codes"])
ligand_lines = [l for l in lines if l.startswith("HETATM") and l[17:20].strip() == target_info["ligand_code"]]
coords = [(float(l[30:38]), float(l[38:46]), float(l[46:54])) for l in ligand_lines]
box_center = [sum(c[i] for c in coords) / len(coords) for i in range(3)]

protonated_tests = {
    "dopamine_protonated": "[NH3+]CCc1ccc(O)c(O)c1",
    "methoxydopamine_protonated": "COc1ccc(CC[NH3+])cc1O",
}

for label, smi in protonated_tests.items():
    scores = []
    for seed in [1, 2, 3, 42, 99]:
        lig_pdbqt = prepare_ligand_pdbqt(smi, f"targets/test_{label}_{seed}", seed=seed)
        if lig_pdbqt is None:
            print(f"{label} seed={seed}: 리간드 준비 실패")
            continue
        score = run_docking_cli(lig_pdbqt, receptor_pdbqt, f"targets/test_{label}_{seed}",
                                  box_center, target_info["box_size"], exhaustiveness=16)
        scores.append(score)
    valid = [s for s in scores if s is not None]
    avg = sum(valid) / len(valid) if valid else None
    print(f"{label}: {scores}, 평균={avg}")

dopamine_protonated: [-5.668, -5.7, -5.64, -5.615, -5.591], 평균=-5.642799999999999
methoxydopamine_protonated: [-5.675, -5.811, -5.857, -5.85, -5.748], 평균=-5.7882


In [13]:
from src.tools.docking import inspect_hetatm

hetero = inspect_hetatm("1VID")
print(hetero)

{'MG', 'HOH', 'DNC', 'SAM'}


In [15]:
importlib.reload(src.tools.docking)
from src.tools.docking import prepare_receptor, prepare_ligand_pdbqt, run_docking_cli, DOCKING_TARGETS

target_info = DOCKING_TARGETS["catechol"]
# 캐시된 receptor_pdbqt가 남아있으면 SAM 제거 전 버전을 계속 쓰게 되니, 강제로 새로 만들도록 기존 파일 삭제
import os
for f in ["targets/1VID_clean.pdb", "targets/1VID_receptor.pdbqt"]:
    if os.path.exists(f):
        os.remove(f)

receptor_pdbqt, lines = prepare_receptor(target_info["pdb_id"], target_info["keep_hetatm_codes"])
ligand_lines = [l for l in lines if l.startswith("HETATM") and l[17:20].strip() == target_info["ligand_code"]]
coords = [(float(l[30:38]), float(l[38:46]), float(l[46:54])) for l in ligand_lines]
box_center = [sum(c[i] for c in coords) / len(coords) for i in range(3)]
print("box_center:", box_center)

for label, smi in [("dopamine", "NCCc1ccc(O)c(O)c1"), ("methoxydopamine", "COc1ccc(CCN)cc1O")]:
    scores = []
    for seed in [1, 2, 3, 42, 99]:
        lig_pdbqt = prepare_ligand_pdbqt(smi, f"targets/test2_{label}_{seed}", seed=seed)
        score = run_docking_cli(lig_pdbqt, receptor_pdbqt, f"targets/test2_{label}_{seed}",
                                  box_center, target_info["box_size"], exhaustiveness=16)
        scores.append(score)
    valid = [s for s in scores if s is not None]
    avg = sum(valid) / len(valid) if valid else None
    print(f"{label}: {scores}, 평균={avg}")

box_center: [-24.806714285714285, 59.999857142857145, 50.56657142857143]
dopamine: [-5.144, -5.153, -5.179, -5.172, -5.173], 평균=-5.164199999999999
methoxydopamine: [-5.194, -5.173, -5.17, -5.172, -5.164], 평균=-5.1746


In [16]:
from src.tools.docking import _load_cache, _save_cache

cache = _load_cache()
removed = [k for k in cache if k.startswith("catechol|")]
for k in removed:
    del cache[k]
_save_cache(cache)
print(f"제거된 캐시 항목: {removed}")

제거된 캐시 항목: ['catechol|NCCc1ccc(O)c(O)c1|COc1ccc(CCN)cc1O']


In [17]:
!cd /content/laidd-2026 && git add -A && git commit -m "Fix DOCKING_TARGETS: catechol/COMT must retain SAM cofactor (was inverting docking direction)" && git push

[main 7dfda12] Fix DOCKING_TARGETS: catechol/COMT must retain SAM cofactor (was inverting docking direction)
 32 files changed, 1412 insertions(+), 11 deletions(-)
 create mode 100644 targets/test2_dopamine_1_log.txt
 create mode 100644 targets/test2_dopamine_2_log.txt
 create mode 100644 targets/test2_dopamine_3_log.txt
 create mode 100644 targets/test2_dopamine_42_log.txt
 create mode 100644 targets/test2_dopamine_99_log.txt
 create mode 100644 targets/test2_methoxydopamine_1_log.txt
 create mode 100644 targets/test2_methoxydopamine_2_log.txt
 create mode 100644 targets/test2_methoxydopamine_3_log.txt
 create mode 100644 targets/test2_methoxydopamine_42_log.txt
 create mode 100644 targets/test2_methoxydopamine_99_log.txt
 create mode 100644 targets/test_dopamine_1_log.txt
 create mode 100644 targets/test_dopamine_2_log.txt
 create mode 100644 targets/test_dopamine_3_log.txt
 create mode 100644 targets/test_dopamine_42_log.txt
 create mode 100644 targets/test_dopamine_99_log.txt
 crea

In [19]:
importlib.reload(src.tools.docking)
from src.tools.docking import auto_dock_precedent, inspect_hetatm, DOCKING_TARGETS

hetero = inspect_hetatm("1DXO")
print("1DXO HETATM 목록:", hetero)  # FAD, DQN 확인용

1DXO HETATM 목록: {'HOH', 'FAD', 'DQN'}


In [21]:
result = auto_dock_precedent(
    "hydroquinone",
    original_smiles="CC1=C(C)C(=O)C(C)=C(C)C1=O",   # 듀로퀴논
    fixed_smiles="CC1=C(C)C(O)=C(C)C(C)=C1O",         # 듀로하이드로퀴논(환원형)
)
print(result)

{'target': 'NQO1', 'pdb_id': '1DXO', 'rule': 'hydroquinone', 'score_original': -4.128, 'score_fixed': -4.316, 'delta': -0.18799999999999972}


In [22]:
!cd /content/laidd-2026 && git add -A && git commit -m "Add NQO1 (1DXO) target for hydroquinone/quinone_A(370); reproduces expected binding direction" && git push

[main 71b2764] Add NQO1 (1DXO) target for hydroquinone/quinone_A(370); reproduces expected binding direction
 4 files changed, 108 insertions(+), 5 deletions(-)
 create mode 100644 targets/hydroquinone_fixed_log.txt
 create mode 100644 targets/hydroquinone_original_log.txt
Enumerating objects: 17, done.
Counting objects: 100% (17/17), done.
Delta compression using up to 2 threads
Compressing objects: 100% (10/10), done.
Writing objects: 100% (10/10), 1.92 KiB | 245.00 KiB/s, done.
Total 10 (delta 6), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (6/6), completed with 5 local objects.
To https://github.com/Dec32th/laidd-2026.git
   7dfda12..71b2764  main -> main


In [23]:
result = auto_dock_precedent(
    "Michael_acceptor_1",
    original_smiles="COC1=CC(=C(C=C1NC2=NC=CC(=N2)C3=CN(C4=CC=CC=C43)C)OC)NC(=O)C=CCN(C)C",  # 오시메르티닙
    fixed_smiles="COC1=CC(=C(C=C1NC2=NC=CC(=N2)C3=CN(C4=CC=CC=C43)C)OC)NC(=O)CCCN(C)C",       # C=C 환원(공유결합 반응성 제거) 버전
)
print(result)

{'target': 'EGFR', 'pdb_id': '6JX4', 'rule': 'Michael_acceptor_1', 'score_original': -7.624, 'score_fixed': -7.382, 'delta': 0.242}


In [24]:
!cd /content/laidd-2026 && git add -A && git commit -m "Verify Michael_acceptor_1/EGFR docking reproduces expected near-zero delta (covalent mechanism blind spot confirmed)" && git push

[main 58e307c] Verify Michael_acceptor_1/EGFR docking reproduces expected near-zero delta (covalent mechanism blind spot confirmed)
 3 files changed, 102 insertions(+)
 create mode 100644 targets/Michael_acceptor_1_fixed_log.txt
 create mode 100644 targets/Michael_acceptor_1_original_log.txt
Enumerating objects: 11, done.
Counting objects: 100% (11/11), done.
Delta compression using up to 2 threads
Compressing objects: 100% (7/7), done.
Writing objects: 100% (7/7), 1.81 KiB | 464.00 KiB/s, done.
Total 7 (delta 4), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (4/4), completed with 3 local objects.
To https://github.com/Dec32th/laidd-2026.git
   71b2764..58e307c  main -> main


In [25]:
!cat src/tools/agent.py


import json
from src.tools.replacement_library import get_replacement_candidates

_llm_error_log = []
_llm_consecutive_failures = 0
_LLM_FAILURE_LIMIT = 3
_debate_call_budget = {"remaining": 100}

def set_debate_budget(n):
    """토의(debate)에 쓸 수 있는 총 LLM 호출 수 상한을 재설정."""
    _debate_call_budget["remaining"] = n

def _call_llm(client, model_name, prompt, client_type="gemini"):
    """client_type에 따라 Gemini SDK 또는 OpenAI 호환 SDK로 호출하고,
    응답 텍스트만 통일된 형태로 반환."""
    if client_type == "gemini":
        response = client.models.generate_content(model=model_name, contents=prompt)
        return response.text
    elif client_type == "openai_compatible":
        for attempt in range(2):  # rate limit 시 1회만 재시도
            try:
                response = client.chat.completions.create(
                    model=model_name,
                    messages=[{"role": "user", "content": prompt}],
                    max_tokens=500,
                    timeout=30,
                    extra_body={"enab

In [40]:
%%writefile src/tools/agent.py
import json
import time
from src.tools.replacement_library import get_replacement_candidates

_llm_error_log = []
_llm_consecutive_failures = 0
_LLM_FAILURE_LIMIT = 3
_debate_call_budget = {"remaining": 100}

def set_debate_budget(n):
    """토의(debate)에 쓸 수 있는 총 LLM 호출 수 상한을 재설정."""
    _debate_call_budget["remaining"] = n

_injected_sascorer = {"module": None}
_injected_tox_predictor = {"fn": None}

def set_sascorer_module(module):
    """세션마다 다운로드/import한 sascorer 모듈을 등록."""
    _injected_sascorer["module"] = module

def set_tox_predictor(fn):
    """(original_smiles, fixed_smiles, rule_name) -> tox_delta(float) 또는 None
    을 반환하는 콜백을 등록. 노트북마다 다르게 학습한 baseline 모델을
    감싸서 넘기면 됨."""
    _injected_tox_predictor["fn"] = fn

def _call_llm(client, model_name, prompt, client_type="gemini"):
    """client_type에 따라 Gemini SDK 또는 OpenAI 호환 SDK로 호출하고,
    응답 텍스트만 통일된 형태로 반환."""
    if client_type == "gemini":
        response = client.models.generate_content(model=model_name, contents=prompt)
        return response.text
    elif client_type == "openai_compatible":
        for attempt in range(2):  # rate limit 시 1회만 재시도
            try:
                response = client.chat.completions.create(
                    model=model_name,
                    messages=[{"role": "user", "content": prompt}],
                    max_tokens=500,
                    timeout=30,
                    extra_body={"enable_thinking": False},
                )
                return response.choices[0].message.content
            except Exception as e:
                _llm_error_log.append(repr(e))
                if 'RateLimitError' in type(e).__name__ and attempt == 0:
                    time.sleep(3)
                    continue
                return f"ERROR: LLM 호출 실패/타임아웃 - {e}"
    else:
        raise ValueError(f"알 수 없는 client_type: {client_type}")


def _parse_json_response(text, fallback):
    text = text.strip()
    if text.startswith('```'):
        text = text.split('```')[1]
        if text.startswith('json'):
            text = text[4:]
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return fallback

def _try_get_docking_evidence(rule_name, smiles_before, smiles_after):
    """도킹 표적이 등록된 규칙이면 자동으로 도킹 실행, 아니면 None.
    도킹 실패/미등록/예외는 전부 조용히 None으로 처리(critic 프롬프트에서
    도킹 근거 없이 진행하는 것으로 자연스럽게 폴백)."""
    try:
        from src.tools.docking import auto_dock_precedent, DOCKING_TARGETS
        if rule_name not in DOCKING_TARGETS:
            return None
        result = auto_dock_precedent(rule_name, smiles_before, smiles_after)
        if result.get('error') or result.get('delta') is None:
            return None
        return result
    except Exception:
        return None

def _try_compute_score(rule_name, smiles_before, smiles_after, docking_evidence=None):
    """등록된 sascorer/tox_predictor/도킹 결과를 모아 종합 점수 계산.
    일부만 등록돼 있어도 compute_multi_objective_score가 나머지로
    자동 정규화하므로 실패하지 않음. 계산 자체가 실패하면 None."""
    try:
        from src.tools.scoring import compute_multi_objective_score

        tox_delta = None
        if _injected_tox_predictor["fn"] is not None:
            try:
                tox_delta = _injected_tox_predictor["fn"](smiles_before, smiles_after, rule_name)
            except Exception:
                tox_delta = None

        precedent_docking_delta = docking_evidence["delta"] if docking_evidence else None

        return compute_multi_objective_score(
            smiles_before, smiles_after, rule_name,
            tox_delta=tox_delta,
            sascorer_module=_injected_sascorer["module"],
            precedent_docking_delta=precedent_docking_delta,
        )
    except Exception:
        return None

def ask_llm_which_problem_to_fix(client, model_name, smiles, problems, client_type="gemini"):
    """여러 toxicophore 중 어떤 것부터 고칠지 LLM에게 판단을 요청."""
    known = [p for p in problems if get_replacement_candidates(p['rule_name']) is not None]

    if not known:
        return None
    if len(known) == 1:
        return {"rule_name": known[0]['rule_name'], "reason": "유일한 치환 가능 후보"}

    prompt = f"""당신은 신약개발 화학자입니다. 다음 분자에서 여러 구조적 문제(toxicophore)가 발견되었습니다.

분자 SMILES: {smiles}

발견된 문제 중, 우리가 실제로 치환 가능한 것들:
{json.dumps(known, ensure_ascii=False, indent=2)}

이 중 어떤 문제를 먼저 해결하는 것이 화학적으로 더 타당한지 판단하고,
반드시 아래 JSON 형식으로만 답하세요. 다른 설명 없이 JSON만 출력하세요.

{{"rule_name": "선택한 문제의 rule_name", "reason": "선택 이유 한 문장"}}
"""

    text = _call_llm(client, model_name, prompt, client_type)
    fallback = {"rule_name": known[0]['rule_name'], "reason": "JSON 파싱 실패, 기본값(첫 번째 후보) 사용"}
    return _parse_json_response(text, fallback)


def ask_llm_which_candidate_to_use(client, model_name, smiles, rule_name, client_type="gemini"):
    """한 문제(rule_name)에 대한 여러 치환 후보 중 어떤 걸 쓸지 LLM에게 판단 요청.

    candidate의 rationale 중 하나라도 '[참고]'로 시작하는 문구가 있으면,
    이는 실제 승인약물 사례에서 이 골격이 안전하게 쓰인 경우가 있다는 뜻이므로,
    candidate가 1개뿐이더라도(원래는 LLM 호출을 건너뛰던 경우) 반드시 LLM에게
    판단을 맡긴다. 이 경우 LLM은 candidate_idx로 -1을 반환하여 "치환을
    보류하고 사람(연구자) 검토가 필요하다"고 명시적으로 표시할 수 있다.
    """
    info = get_replacement_candidates(rule_name)
    if info is None:
        return None

    candidates = info['candidates']
    has_caution = any('[참고]' in c.get('rationale', '') for c in candidates)

    if len(candidates) == 1 and not has_caution:
        return {"candidate_idx": 0, "reason": "유일한 후보"}

    candidate_info = [
        {"idx": i, "name": c['name'], "rationale": c['rationale']}
        for i, c in enumerate(candidates)
    ]

    prompt = f"""당신은 신약개발 화학자입니다. 다음 분자에서 '{rule_name}' 문제를
해결하기 위한 치환 후보가 있습니다.

분자 SMILES: {smiles}

치환 후보들:
{json.dumps(candidate_info, ensure_ascii=False, indent=2)}

각 후보의 rationale에 "[참고]"로 시작하는 문구가 있다면, 이는 "이 골격이
실제 승인 약물에서 반응성이 아닌 안정적 형태로 널리 쓰인 사례가 있으니,
경고를 절대적 기준이 아닌 참고 신호로 해석하라"는 뜻입니다. 이 경우 먼저
"이 분자가 그 참고사항이 가리키는 안전한 사용 사례와 실제로 유사한지"를
판단하세요.
- 유사하다고 판단되면서, 후보가 여러 개라면 변화 폭이 더 작은 후보를 선택하세요.
- 유사하다고 판단되고, 치환 자체가 불필요하다고 볼 만큼 뚜렷하다면,
  candidate_idx를 -1로 답해 "치환 보류, 사람 검토 필요"를 표시하세요.
- 참고사항이 없거나 이 분자가 그 사례와 유사하지 않다면, 평소대로 가장
  적절한 후보를 선택하세요.

반드시 아래 JSON 형식으로만 답하세요. 다른 설명 없이 JSON만 출력하세요.

{{"candidate_idx": 선택한 후보의 idx(정수, 또는 보류 시 -1), "reason": "판단 이유 한 문장"}}
"""

    text = _call_llm(client, model_name, prompt, client_type)
    fallback = {"candidate_idx": 0, "reason": "JSON 파싱 실패, 기본값(첫 번째 후보) 사용"}
    result = _parse_json_response(text, fallback)

    idx = result.get('candidate_idx')
    if not isinstance(idx, int) or not (-1 <= idx < len(candidates)):
        return {"candidate_idx": 0, "reason": "LLM 응답 idx 범위 오류, 기본값 사용"}
    return result


def ask_llm_debate_fix(client, model_name, smiles_before, smiles_after, rule_name,
                        candidate_name, candidate_rationale, client_type="gemini",
                        max_rounds=2):
    """제안자(원래 candidate를 고른 논리)와 검토자(critic)가 여러 라운드
    대화하며 합의에 도달하려 시도. 매 라운드 critic이 판단하고, 반려하면
    proposer가 반박, critic이 재판단. max_rounds 안에 합의(양쪽 다 승인,
    또는 critic이 최종 반려로 확정) 안 되면 "escalate"로 사람 검토行.

    반환: {"final_verdict": "approved"|"rejected"|"escalate",
           "rounds": [{"role": "critic"|"proposer", "text": str}, ...],
           "consensus_reached": bool}
    """
    if _debate_call_budget["remaining"] <= 0:
        return {"final_verdict": "approved", "rounds": [], "consensus_reached": True,
                "budget_exhausted": True}
    _debate_call_budget["remaining"] -= 1
    rounds_log = []
    proposer_argument = candidate_rationale

    for round_num in range(1, max_rounds + 1):
        docking_evidence = _try_get_docking_evidence(rule_name, smiles_before, smiles_after)
        score_result = _try_compute_score(rule_name, smiles_before, smiles_after, docking_evidence)
        critic_prompt = f"""당신은 신약개발 화학 검토자(critic)입니다. 동료 화학자가 아래
치환을 제안했습니다.

원본 분자: {smiles_before}
치환 후 분자: {smiles_after}
해결하려던 문제: {rule_name}
제안된 치환: {candidate_name}
제안자의 근거: {proposer_argument}
{f"실측 도킹 결합력 변화: {docking_evidence['target']} 표적, {docking_evidence['score_original']:.2f} → {docking_evidence['score_fixed']:.2f} kcal/mol (delta {docking_evidence['delta']:+.2f}). 이 정량 데이터를 판단에 반영하세요." if docking_evidence else ""}
{f"종합 점수: {score_result['composite_score']:.2f} (세부: {score_result['component_scores']}). 이것도 판단에 참고하세요." if score_result and score_result.get('composite_score') is not None else ""}

이 치환에 동의하는지 비판적으로 검토하세요. 동의하지 않는다면 구체적으로
어떤 점이 문제인지 명시하세요(새로운 독성 구조 생성 가능성, 근거의
논리적 결함, precedent 오독 등).

반드시 아래 JSON 형식으로만 답하세요.
{{"verdict": "approved" 또는 "rejected", "reason": "판단 이유, 반려 시 구체적 반론 포함"}}
"""
        critic_text = _call_llm(client, model_name, critic_prompt, client_type)
        critic_result = _parse_json_response(
            critic_text, {"verdict": "approved", "reason": "JSON 파싱 실패, 기본 승인"}
        )
        rounds_log.append({"role": "critic", "round": round_num, "text": critic_result})

        if critic_result.get("verdict") == "approved":
            return {"final_verdict": "approved", "rounds": rounds_log, "consensus_reached": True}

        if round_num == max_rounds:
            break

        proposer_prompt = f"""당신은 방금 아래 치환을 제안한 화학자입니다.

원본 분자: {smiles_before}
치환 후 분자: {smiles_after}
당신의 원래 근거: {proposer_argument}

동료 검토자(critic)가 다음과 같이 반려했습니다: "{critic_result.get('reason', '')}"

이 반론에 대해 답하세요. 반론이 타당하면 인정하고 제안을 철회하세요.
반론이 부당하다면 왜 원래 치환이 여전히 타당한지 반박하세요.

반드시 아래 JSON 형식으로만 답하세요.
{{"stance": "withdraw" 또는 "defend", "argument": "반박 또는 철회 이유"}}
"""
        proposer_text = _call_llm(client, model_name, proposer_prompt, client_type)
        proposer_result = _parse_json_response(
            proposer_text, {"stance": "withdraw", "argument": "JSON 파싱 실패, 기본 철회"}
        )
        rounds_log.append({"role": "proposer", "round": round_num, "text": proposer_result})

        if proposer_result.get("stance") == "withdraw":
            return {"final_verdict": "rejected", "rounds": rounds_log, "consensus_reached": True}

        proposer_argument = proposer_result.get("argument", proposer_argument)

    return {"final_verdict": "escalate", "rounds": rounds_log, "consensus_reached": False}
def should_debate(candidate_rationale):
    """이 candidate가 토의(debate)를 거칠 필요가 있는지 판단.
    rationale에 '[참고]'가 있으면 실제 승인약물 사례와 겹칠 수 있다는
    뜻이므로, 단순 채택 대신 토의로 한 번 더 검토해야 함."""
    return '[참고]' in (candidate_rationale or '')


Overwriting src/tools/agent.py


In [36]:
import ast
with open('src/tools/agent.py') as f:
    content = f.read()
    ast.parse(content)
print("✅ 문법 정상")
print("✅ import time 반영:", 'import time' in content)
print("✅ _try_get_docking_evidence 반영:", '_try_get_docking_evidence' in content)

importlib.reload(src.tools.agent)
from src.tools.agent import ask_llm_debate_fix

class RecordingMockClient:
    class _Choice:
        def __init__(self, content):
            self.message = type('obj', (), {'content': content})
    class _Response:
        def __init__(self, content):
            self.choices = [RecordingMockClient._Choice(content)]
    class _Completions:
        def __init__(self, canned):
            self.canned = canned
            self.log = []
        def create(self, **kwargs):
            self.log.append(kwargs['messages'][0]['content'])
            return RecordingMockClient._Response(self.canned)
    class _Chat:
        def __init__(self, canned):
            self.completions = RecordingMockClient._Completions(canned)
    def __init__(self, canned):
        self.chat = RecordingMockClient._Chat(canned)

mock = RecordingMockClient('{"verdict": "approved", "reason": "타당함"}')

r = ask_llm_debate_fix(
    mock, "mock", "NCCc1ccc(O)c(O)c1", "COc1ccc(CCN)cc1O", "catechol",
    "methylated catechol", "[참고] 카테콜 골격은...", client_type="openai_compatible",
)

log = mock.chat.completions.log
print("호출 횟수:", len(log))
if log:
    print("critic에게 전달된 프롬프트에 '실측 도킹' 포함 여부:", "실측 도킹" in log[0])
print(r['final_verdict'])

✅ 문법 정상
✅ import time 반영: True
✅ _try_get_docking_evidence 반영: True
호출 횟수: 1
critic에게 전달된 프롬프트에 '실측 도킹' 포함 여부: True
approved


In [37]:
!cd /content/laidd-2026 && git add -A && git commit -m "Connect docking evidence into debate critic prompt (auto-triggered for registered DOCKING_TARGETS)" && git push

[main b088744] Connect docking evidence into debate critic prompt (auto-triggered for registered DOCKING_TARGETS)
 4 files changed, 46 insertions(+), 21 deletions(-)
Enumerating objects: 19, done.
Counting objects: 100% (19/19), done.
Delta compression using up to 2 threads
Compressing objects: 100% (10/10), done.
Writing objects: 100% (10/10), 1.83 KiB | 468.00 KiB/s, done.
Total 10 (delta 8), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (8/8), completed with 8 local objects.
To https://github.com/Dec32th/laidd-2026.git
   58e307c..b088744  main -> main


In [38]:
import inspect
from src.tools.scoring import compute_multi_objective_score
print(inspect.getsource(compute_multi_objective_score))

def compute_multi_objective_score(original_smiles, fixed_smiles, rule_name,
                                    tox_delta=None, sascorer_module=None,
                                    precedent_docking_delta=None,
                                    weights=None):
    """0~1 범위로 정규화한 항목별 점수와 가중합을 반환.
    tox_delta: 외부에서 baseline 모델로 계산한 독성 예측값 변화(음수=개선),
               없으면 None으로 두고 해당 항목 제외.
    sascorer_module: sascorer 모듈(외부에서 import해서 전달, 순환import 방지).
    precedent_docking_delta: 선례 라이브러리에 해당 규칙의 도킹 kcal/mol
               변화값이 있으면 전달(음수=결합강화), 없으면 None.
    weights: 항목별 가중치 딕셔너리, 기본값 아래 참고."""
    default_weights = {"toxicity": 0.30, "docking": 0.20, "sa": 0.15,
                        "qed": 0.15, "lipinski": 0.10, "pains": 0.10}
    w = weights or default_weights

    mol_o = Chem.MolFromSmiles(original_smiles)
    mol_f = Chem.MolFromSmiles(fixed_smiles)
    if mol_o is None or mol_f is None:
        return None

    scores = {}
    used_weight = 0.0

    if tox_delta is not

In [41]:
import ast
with open('src/tools/agent.py') as f:
    content = f.read()
    ast.parse(content)
print("✅ 문법 정상")
print("✅ set_sascorer_module/set_tox_predictor 반영:", 'set_sascorer_module' in content and 'set_tox_predictor' in content)
print("✅ _try_compute_score 반영:", '_try_compute_score' in content)

importlib.reload(src.tools.agent)
from src.tools.agent import ask_llm_debate_fix

# sascorer/tox_predictor 아무것도 등록 안 한 상태 -> QED/Lipinski/PAINS만으로도 정상 동작해야 함
mock = RecordingMockClient('{"verdict": "approved", "reason": "타당함"}')
r = ask_llm_debate_fix(
    mock, "mock", "NCCc1ccc(O)c(O)c1", "COc1ccc(CCN)cc1O", "catechol",
    "methylated catechol", "[참고] 카테콜 골격은...", client_type="openai_compatible",
)
log = mock.chat.completions.log
print("종합 점수 포함 여부:", "종합 점수" in log[0])
print(r['final_verdict'])

✅ 문법 정상
✅ set_sascorer_module/set_tox_predictor 반영: True
✅ _try_compute_score 반영: True
종합 점수 포함 여부: True
approved


In [42]:
%%writefile models/tox_baseline.py
"""Tox21 baseline 독성 예측 모델. 매 세션 직접 학습해서 사용(random_state
고정으로 재현성 확보). debate 로직의 tox_delta 콜백에 연결하기 위한 래퍼."""

import numpy as np
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator
from sklearn.ensemble import RandomForestClassifier

_generator = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)


def smiles_to_ecfp(smiles):
    mol = Chem.MolFromSmiles(smiles)
    return _generator.GetFingerprintAsNumPy(mol) if mol else None


def train_tox21_baseline(data):
    """data: load_tox21_clean() 반환 딕셔너리.
    반환: {"classifiers": dict, "task_cols": list} — 이 딕셔너리 자체가
    학습된 모델 전체이며, 세션 내내 이 변수 하나만 들고 다니면 됨."""
    X_train, y_train, w_train = data['X_train'], data['y_train'], data['w_train']
    task_cols = data['task_cols']
    classifiers = {}
    for i, task in enumerate(task_cols):
        train_mask = w_train[:, i] == 1
        clf = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
        clf.fit(X_train[train_mask], y_train[train_mask, i])
        classifiers[task] = clf
    return {"classifiers": classifiers, "task_cols": task_cols}


def predict_tox21_avg(smiles, model):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    fp = smiles_to_ecfp(smiles).reshape(1, -1)
    classifiers, task_cols = model["classifiers"], model["task_cols"]
    return float(np.mean([classifiers[t].predict_proba(fp)[0][1] for t in task_cols]))


def make_tox_predictor(model):
    """set_tox_predictor()에 바로 넘길 수 있는 콜백 생성.
    반환값은 (fixed 예측 - original 예측): 음수면 독성이 줄어든 것."""
    def _predict(original_smiles, fixed_smiles, rule_name):
        p_o = predict_tox21_avg(original_smiles, model)
        p_f = predict_tox21_avg(fixed_smiles, model)
        if p_o is None or p_f is None:
            return None
        return p_f - p_o
    return _predict

Writing models/tox_baseline.py


In [44]:
import os
init_path = 'models/__init__.py'
if not os.path.exists(init_path):
    open(init_path, 'w').close()
    print("models/__init__.py 생성함")
else:
    print("models/__init__.py 이미 있음")

import ast
with open('models/tox_baseline.py') as f:
    ast.parse(f.read())
print("✅ 문법 정상")

import models.tox_baseline
importlib.reload(models.tox_baseline)
from models.tox_baseline import train_tox21_baseline, make_tox_predictor
print("✅ import 성공")

models/__init__.py 이미 있음
✅ 문법 정상
✅ import 성공


In [48]:
import urllib.request, sys
urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/rdkit/rdkit/master/Contrib/SA_Score/sascorer.py", "sascorer.py")
urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/rdkit/rdkit/master/Contrib/SA_Score/fpscores.pkl.gz", "fpscores.pkl.gz")
sys.path.append('.')
import sascorer

from src.tools.agent import set_sascorer_module
set_sascorer_module(sascorer)
print("sascorer 등록 완료")

sascorer 등록 완료


In [49]:
tox_model = train_tox21_baseline(data)

from models.tox_baseline import make_tox_predictor
tox_predictor = make_tox_predictor(tox_model)

from src.tools.agent import set_tox_predictor
set_tox_predictor(tox_predictor)
print("tox_predictor 등록 완료")

from src.tools.agent import ask_llm_debate_fix
mock = RecordingMockClient('{"verdict": "approved", "reason": "타당함"}')
r = ask_llm_debate_fix(
    mock, "mock", "NCCc1ccc(O)c(O)c1", "COc1ccc(CCN)cc1O", "catechol",
    "methylated catechol", "[참고] 카테콜 골격은...", client_type="openai_compatible",
)
log = mock.chat.completions.log
print(log[0])

tox_predictor 등록 완료
당신은 신약개발 화학 검토자(critic)입니다. 동료 화학자가 아래
치환을 제안했습니다.

원본 분자: NCCc1ccc(O)c(O)c1
치환 후 분자: COc1ccc(CCN)cc1O
해결하려던 문제: catechol
제안된 치환: methylated catechol
제안자의 근거: [참고] 카테콜 골격은...
실측 도킹 결합력 변화: COMT 표적, -5.18 → -5.19 kcal/mol (delta -0.01). 이 정량 데이터를 판단에 반영하세요.
종합 점수: 0.66 (세부: {'toxicity': 0.53, 'docking': 0.5050000000000003, 'sa': 1.0, 'qed': 0.6580250816015243, 'lipinski': 0.5, 'pains': 1.0}). 이것도 판단에 참고하세요.

이 치환에 동의하는지 비판적으로 검토하세요. 동의하지 않는다면 구체적으로
어떤 점이 문제인지 명시하세요(새로운 독성 구조 생성 가능성, 근거의
논리적 결함, precedent 오독 등).

반드시 아래 JSON 형식으로만 답하세요.
{"verdict": "approved" 또는 "rejected", "reason": "판단 이유, 반려 시 구체적 반론 포함"}



In [51]:
!cd /content/laidd-2026 && git add -A && git commit -m "Add Tox21 baseline model (models/tox_baseline.py) and connect to debate critic prompt via set_tox_predictor" && git push

[main 591076d] Add Tox21 baseline model (models/tox_baseline.py) and connect to debate critic prompt via set_tox_predictor
 3 files changed, 89 insertions(+)
 create mode 100644 models/__init__.py
 create mode 100644 models/tox_baseline.py
Enumerating objects: 12, done.
Counting objects: 100% (12/12), done.
Delta compression using up to 2 threads
Compressing objects: 100% (7/7), done.
Writing objects: 100% (7/7), 2.47 KiB | 631.00 KiB/s, done.
Total 7 (delta 3), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (3/3), completed with 3 local objects.
To https://github.com/Dec32th/laidd-2026.git
   b088744..591076d  main -> main
